# Figure 4

This notebook prepares and visualises the empirical-landscape ruggedness analyses for Figure 4. It reads the raw empirical decay products generated by `data_generation.ipynb`, writes Figure-4-specific processed payloads into `processed_data/`, and saves the composed figure into `figures/pdf/`, `figures/png/`, and `figures/eps/`.


## Setup

Load the empirical-landscape, decay-fitting, and plotting tools used by the standalone Figure 4 workflow.


In [ ]:
%load_ext autoreload
%autoreload 2

import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec
from tqdm.auto import tqdm

from slide.data_generation import EMPIRICAL_NAMES, RAW_FILENAMES
from slide.data_processing import load_landscapes
from slide.direvo_functions import get_single_decay_rate_IK_v2
from slide.ruggedness_functions import (
    find_distance_to_closest_max,
    get_dirichlet_metric,
    get_landscape_spectrum,
    get_mean_paths_to_max,
    get_spectral_entropy,
    landscape_r2,
    local_epistasis,
    max_possible_paths,
    roughness_to_slope,
)
from slide.utils import get_figures_dir, get_processed_data_dir, get_raw_data_dir, load_pickle, save_pickle

OVERWRITE_PROCESSED_PKL: bool = False
SAVE_FIGURES: bool = True
SAVE_TYPE_LIST = ("pdf", "png", "eps")
PANEL_DPI = 350

RAW_DATA_DIR = get_raw_data_dir()
PROCESSED_DATA_DIR = get_processed_data_dir()
FIGURES_DIR = get_figures_dir()
for figure_type in SAVE_TYPE_LIST:
    (FIGURES_DIR / figure_type).mkdir(parents=True, exist_ok=True)

print(f"raw_data: {RAW_DATA_DIR}")
print(f"processed_data: {PROCESSED_DATA_DIR}")
print(f"figures: {FIGURES_DIR}")



def save_panel_figure(fig: plt.Figure, stem: str, *, bbox_inches: str = "tight") -> None:
    """Save an individual panel figure in every configured output format."""
    if not SAVE_FIGURES:
        return
    for figure_type in SAVE_TYPE_LIST:
        fig.savefig(FIGURES_DIR / figure_type / f"{stem}.{figure_type}", dpi=PANEL_DPI, bbox_inches=bbox_inches)


def add_panel_letter(ax: plt.Axes, letter: str) -> None:
    """Add the panel letter annotation used in the paper figures."""
    ax.text(-0.16, 1.08, letter, transform=ax.transAxes, fontsize=12, fontweight="bold", va="top", ha="left")


## Figure 4 Files And Parameters

Define the raw products consumed by Figure 4 and the Figure-4-specific processed filenames. Raw products are expected to come from `data_generation.ipynb`; this notebook does not regenerate the large all-start empirical diffusion simulations.


In [ ]:
RAW_KEYS = {
    "empirical_all": [f"empirical_decay_{name}_all" for name in EMPIRICAL_NAMES],
    "empirical_popsize": [f"empirical_decay_{name}_popsize" for name in EMPIRICAL_NAMES],
}

PROCESSED_FILES = {
    "spectra": "figure4_empirical_fourier_spectra_processed.pkl",
    "local_rho": "figure4_local_rho_processed.pkl",
    "replicate_variability": "figure4_replicate_variability_processed.pkl",
    "subsampling": "figure4_subsampling_processed.pkl",
    "popsize": "figure4_popsize_processed.pkl",
    "generation_count": "figure4_generation_count_processed.pkl",
    "metric_comparison": "figure4_metric_comparison_processed.pkl",
}

# Processing controls. Set MAX_LOCAL_STARTS to an integer for a quick smoke run;
# use None for the full all-start Figure 4B calculation.
MAX_LOCAL_STARTS: int | None = None
REPLICATE_VARIABILITY_BOOTSTRAPS = 250
SUBSAMPLING_BOOTSTRAPS = 1000
SUBSAMPLING_POINTS = 11
GENERATION_COUNT_BOOTSTRAPS = 250
POPSIZE_BOOTSTRAPS = 250
BOOTSTRAP_STARTS_FOR_GLOBAL_ESTIMATES = 1000
BOOTSTRAP_SEED = 42
MUTATION_RATE = 0.1
EPS = 1e-8

labels = ["GB1", "TrpB", "TEV", "ParD3"]
colours = ["tab:orange", "tab:blue", "tab:green", "#f55f74"]
markers = ["o", "s", "^", "D"]

required_raw_paths = {
    key: RAW_DATA_DIR / RAW_FILENAMES[key]
    for group in RAW_KEYS.values()
    for key in group
}
missing_raw_paths = {key: path for key, path in required_raw_paths.items() if not path.exists()}
if missing_raw_paths:
    print("Missing raw products needed by Figure 4:")
    for key, path in missing_raw_paths.items():
        print(f"  {key}: {path}")
else:
    print("All Figure 4 raw products are present.")


## Processing Helpers

These small helpers keep the Figure 4 cells readable while preserving the raw array conventions used in the paper pipeline. They accept both raw trajectories that still contain population-level fitness values and newer raw trajectories that already store mean population fitness.


In [ ]:
def processed_path(key: str) -> Path:
    return PROCESSED_DATA_DIR / PROCESSED_FILES[key]


def load_or_build_processed(key: str, builder):
    path = processed_path(key)
    if path.exists() and not OVERWRITE_PROCESSED_PKL:
        print(f"Loaded {path.name}")
        return load_pickle(path)
    payload = builder()
    save_pickle(payload, path)
    print(f"Saved {path.name}")
    return payload


def require_raw_products(keys: list[str]) -> None:
    missing = [key for key in keys if not (RAW_DATA_DIR / RAW_FILENAMES[key]).exists()]
    if missing:
        message = "Missing raw products. Run data_generation.ipynb first for: " + ", ".join(missing)
        raise FileNotFoundError(message)


def load_raw_payload(key: str) -> dict:
    return load_pickle(RAW_DATA_DIR / RAW_FILENAMES[key])


def as_start_rep_curves(decay: np.ndarray) -> np.ndarray:
    arr = np.asarray(decay)
    if arr.ndim == 3:
        return arr.astype(float)
    if arr.ndim == 4:
        return arr.mean(axis=2).astype(float)
    raise ValueError(f"Expected 3D or 4D empirical decay array, got shape {arr.shape}")


def as_popsize_start_rep_curves(decay: np.ndarray) -> np.ndarray:
    arr = np.asarray(decay)
    if arr.ndim == 4:
        return arr.astype(float)
    if arr.ndim == 5:
        return arr.mean(axis=3).astype(float)
    raise ValueError(f"Expected 4D or 5D empirical popsize decay array, got shape {arr.shape}")


def fit_rho2_from_gmu(g_mu: np.ndarray, *, steps: int | None = None, mutation_rate: float = MUTATION_RATE) -> float:
    curve = np.asarray(g_mu, dtype=float)
    if steps is not None:
        curve = curve[: int(steps)]
    if curve.size < 3 or not np.all(np.isfinite(curve)):
        return np.nan
    denom = max(abs(float(curve[0])), EPS)
    normalized = np.clip(curve / denom, EPS, None)
    try:
        return float(get_single_decay_rate_IK_v2(normalized, mut=mutation_rate, num_steps=len(normalized), fix_amplitude=True)[0] / 2)
    except Exception:
        return np.nan


def fit_rho2_from_fmu(f_mu: np.ndarray, *, steps: int | None = None, mutation_rate: float = MUTATION_RATE) -> float:
    curve = np.asarray(f_mu, dtype=float)
    if steps is not None:
        curve = curve[: int(steps)]
    return fit_rho2_from_gmu(curve**2, mutation_rate=mutation_rate)


def finite_array(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    return values[np.isfinite(values)]


## Raw Empirical Payloads

Load the all-start empirical decay products and the population-size decay products needed for Figure 4B-F.


In [ ]:
require_raw_products(RAW_KEYS["empirical_all"] + RAW_KEYS["empirical_popsize"])

landscapes = load_landscapes()
empirical_all_payloads = {
    name: load_raw_payload(f"empirical_decay_{name}_all")
    for name in EMPIRICAL_NAMES
}
empirical_popsize_payloads = {
    name: load_raw_payload(f"empirical_decay_{name}_popsize")
    for name in EMPIRICAL_NAMES
}
empirical_all_curves = {
    name: as_start_rep_curves(payload["data"])
    for name, payload in empirical_all_payloads.items()
}
empirical_popsize_curves = {
    name: as_popsize_start_rep_curves(payload["data"])
    for name, payload in empirical_popsize_payloads.items()
}

for name in EMPIRICAL_NAMES:
    print(name, "all-start", empirical_all_curves[name].shape, "popsize", empirical_popsize_curves[name].shape)


## Panel A Processing

Compute collapsed Fourier spectra for the complete empirical landscapes. The analytical `rho_2` value is the Fourier-power weighted spectral centroid; ParD3 uses `N=3`, while GB1, TrpB, and TEV use `N=4`.


In [ ]:
def build_spectra_payload():
    spectra = [
        get_landscape_spectrum(landscapes[name], remove_constant=False, on_gpu=False, norm=False)
        for name in EMPIRICAL_NAMES
    ]
    A = 20
    dimensions = np.array([landscapes[name].ndim for name in EMPIRICAL_NAMES], dtype=int)
    fourier_vals = []
    frequency_centroids = []
    for n, spectrum in enumerate(spectra):
        nonconstant = np.asarray(spectrum, dtype=float)[1:]
        indexes = np.arange(len(nonconstant)) + 1
        d = dimensions[n] * (A - 1)
        rho2 = np.sum(A * indexes * nonconstant) / np.sum(nonconstant) / d
        fourier_vals.append(float(rho2))
        frequency_centroids.append(float(rho2 * d / A))
    return {
        "data": {
            "spectra": spectra,
            "rho2_fourier": np.asarray(fourier_vals),
            "frequency_centroids": np.asarray(frequency_centroids),
            "dimensions": dimensions,
            "A": A,
            "labels": labels,
        },
        "metadata": {"paper_reference": "Figure 4A", "description": "Empirical Fourier spectra and analytical rho2 values."},
    }

spectra_payload = load_or_build_processed("spectra", build_spectra_payload)
spectra_data = spectra_payload["data"]
fourier_vals = np.asarray(spectra_data["rho2_fourier"])
fourier_vals


### Panel A Individual Export
This cell saves the annotated Figure 4A subfigure after the empirical Fourier spectra have been processed.


In [ ]:
fig, ax = plt.subplots(figsize=(3.3, 2.8), dpi=PANEL_DPI)
for n, spectrum in enumerate(spectra_data["spectra"]):
    nonconstant = np.asarray(spectrum, dtype=float)[1:]
    spectrum_norm = (nonconstant - nonconstant.min()) / (nonconstant.max() - nonconstant.min() + EPS)
    indexes = np.arange(1, len(nonconstant) + 1)
    ax.plot(indexes, spectrum_norm, marker=markers[n], markersize=4, linewidth=1, color=colours[n], label=labels[n])
    ax.axvline(spectra_data["frequency_centroids"][n], color=colours[n], linestyle="--", alpha=0.55, linewidth=1)
ax.set_title("Empirical Fourier spectra")
ax.set_xlabel(r"Frequency index $i$")
ax.set_ylabel(r"Norm. power $b_i$")
ax.set_xticks([1, 2, 3, 4])
ax.legend(fontsize=7, frameon=False)
add_panel_letter(ax, "A")
save_panel_figure(fig, "figure_4A")
plt.show()


## Panel B Processing

Fit local squared-decay estimates from individual starting points. Each starting point is represented by its replicate-averaged mutation-only decay curve.


In [ ]:
def build_local_rho_payload():
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    local_rhos = {}
    used_start_indices = {}
    for name in EMPIRICAL_NAMES:
        curves = empirical_all_curves[name]
        start_curves = curves.mean(axis=1)
        if MAX_LOCAL_STARTS is None or MAX_LOCAL_STARTS >= start_curves.shape[0]:
            indices = np.arange(start_curves.shape[0])
        else:
            indices = np.sort(rng.choice(start_curves.shape[0], size=MAX_LOCAL_STARTS, replace=False))
        vals = [
            fit_rho2_from_fmu(start_curves[idx], steps=start_curves.shape[-1])
            for idx in tqdm(indices, desc=f"{name} local rho2")
        ]
        local_rhos[name] = finite_array(vals)
        used_start_indices[name] = indices
    return {
        "data": {"local_rhos": local_rhos, "used_start_indices": used_start_indices},
        "metadata": {"paper_reference": "Figure 4B", "description": "Local fitted rho2 estimates by empirical starting genotype."},
    }

local_rho_payload = load_or_build_processed("local_rho", build_local_rho_payload)
local_rhos = local_rho_payload["data"]["local_rhos"]
{name: values.shape for name, values in local_rhos.items()}


### Panel B Individual Export
This cell saves the annotated Figure 4B subfigure after the local ruggedness estimates have been processed.


In [ ]:
local_data = [finite_array(local_rhos[name]) for name in EMPIRICAL_NAMES]
fig, ax = plt.subplots(figsize=(3.3, 2.8), dpi=PANEL_DPI)
vp = ax.violinplot(local_data, showmeans=True, showextrema=False)
for body, color in zip(vp["bodies"], colours):
    body.set_facecolor(color)
    body.set_edgecolor("black")
    body.set_alpha(0.55)
vp["cmeans"].set_color("black")
ax.scatter(np.arange(1, len(EMPIRICAL_NAMES) + 1), fourier_vals, color="black", s=20, zorder=3, label=r"Analytical $\rho_2$")
ax.set_xticks(np.arange(1, len(EMPIRICAL_NAMES) + 1))
ax.set_xticklabels(labels, rotation=25, ha="right")
ax.set_ylabel(r"Local fitted $\rho_2$")
ax.set_title("Local ruggedness estimates")
ax.legend(fontsize=7, frameon=False)
add_panel_letter(ax, "B")
save_panel_figure(fig, "figure_4B")
plt.show()


## Panel C Processing

Estimate uncertainty from repeated measurements with 10 starting genotypes. For each bootstrap draw, the same starting genotypes are reused across five replicate estimates, and the standard deviation across those estimates is recorded.


In [ ]:
def build_replicate_variability_payload():
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    variability = {}
    for name in EMPIRICAL_NAMES:
        curves = empirical_all_curves[name]
        n_starts, n_reps, n_steps = curves.shape
        boot_stds = []
        starts_per_boot = min(10, n_starts)
        reps_per_boot = min(5, n_reps)
        for _ in tqdm(range(REPLICATE_VARIABILITY_BOOTSTRAPS), desc=f"{name} replicate variability"):
            start_idx = rng.choice(n_starts, size=starts_per_boot, replace=False)
            rep_idx = rng.choice(n_reps, size=reps_per_boot, replace=False)
            estimates = []
            for rep in rep_idx:
                curve = curves[start_idx, rep, :].mean(axis=0)
                estimates.append(fit_rho2_from_fmu(curve, steps=n_steps))
            estimates = finite_array(estimates)
            if estimates.size:
                boot_stds.append(float(np.std(estimates)))
        variability[name] = finite_array(boot_stds)
    return {
        "data": {"variability": variability, "starts_per_boot": 10, "replicates_per_boot": 5},
        "metadata": {"paper_reference": "Figure 4C", "description": "Bootstrap standard deviation of fitted rho2 estimates."},
    }

replicate_variability_payload = load_or_build_processed("replicate_variability", build_replicate_variability_payload)
replicate_variability = replicate_variability_payload["data"]["variability"]
{name: values.shape for name, values in replicate_variability.items()}


### Panel C Individual Export
This cell saves the annotated Figure 4C subfigure after the replicate uncertainty summary has been processed.


In [ ]:
variability_data = [finite_array(replicate_variability[name]) for name in EMPIRICAL_NAMES]
fig, ax = plt.subplots(figsize=(3.3, 2.8), dpi=PANEL_DPI)
vp = ax.violinplot(variability_data, showmeans=True, showextrema=False)
for body, color in zip(vp["bodies"], colours):
    body.set_facecolor(color)
    body.set_edgecolor("black")
    body.set_alpha(0.55)
vp["cmeans"].set_color("black")
for i, values in enumerate(variability_data, start=1):
    ax.hlines(np.nanmean(values), i - 0.28, i + 0.28, colors="black", linewidth=1.2)
ax.set_xticks(np.arange(1, len(EMPIRICAL_NAMES) + 1))
ax.set_xticklabels(labels, rotation=25, ha="right")
ax.set_ylabel(r"s.d. of fitted $\rho_2$")
ax.set_title("Replicate uncertainty")
add_panel_letter(ax, "C")
save_panel_figure(fig, "figure_4C")
plt.show()


## Panel D Processing

Bootstrap global `rho_2` estimates while increasing the number of starting trajectories averaged. The largest count corresponds to the full empirical genotype space for each landscape.


In [ ]:
def build_subsampling_payload():
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    subsampling = {}
    trajectory_counts = {}
    for name in EMPIRICAL_NAMES:
        curves = empirical_all_curves[name]
        start_gmu = curves.mean(axis=1) ** 2
        counts = np.unique(np.round(np.logspace(0, np.log10(start_gmu.shape[0]), SUBSAMPLING_POINTS)).astype(int))
        trajectory_counts[name] = counts
        landscape_results = []
        for count in tqdm(counts, desc=f"{name} subsampling"):
            boot_vals = []
            replace = count > start_gmu.shape[0]
            for _ in range(SUBSAMPLING_BOOTSTRAPS):
                idx = rng.choice(start_gmu.shape[0], size=int(count), replace=replace)
                g_mu = start_gmu[idx].mean(axis=0)
                boot_vals.append(fit_rho2_from_gmu(g_mu, steps=g_mu.shape[-1]))
            landscape_results.append(finite_array(boot_vals))
        subsampling[name] = landscape_results
    return {
        "data": {"subsampling": subsampling, "trajectory_counts": trajectory_counts},
        "metadata": {"paper_reference": "Figure 4D", "description": "Global fitted rho2 accuracy over sampled starting trajectories."},
    }

subsampling_payload = load_or_build_processed("subsampling", build_subsampling_payload)
subsampling_data = subsampling_payload["data"]["subsampling"]
trajectory_counts = subsampling_payload["data"]["trajectory_counts"]
{name: [len(v) for v in vals] for name, vals in subsampling_data.items()}


### Panel D Individual Export
This cell saves the annotated Figure 4D subfigure after the starting-point subsampling analysis has been processed.


In [ ]:
fig, ax = plt.subplots(figsize=(3.3, 2.8), dpi=PANEL_DPI)
for n, name in enumerate(EMPIRICAL_NAMES):
    counts = np.asarray(trajectory_counts[name])
    vals = subsampling_data[name]
    means = np.asarray([np.nanmean(v) for v in vals])
    stds = np.asarray([np.nanstd(v) for v in vals])
    ax.plot(counts, means, color=colours[n], marker=markers[n], markersize=4, linewidth=1, label=labels[n])
    ax.fill_between(counts, means - stds, means + stds, color=colours[n], alpha=0.18)
    ax.hlines(fourier_vals[n], xmin=counts.min(), xmax=counts.max(), color=colours[n], linestyle="dotted", linewidth=1)
ax.set_xscale("log")
ax.set_title("Starting-point subsampling")
ax.set_xlabel("Number of starting points")
ax.set_ylabel(r"Global fitted $\rho_2$")
ax.legend(fontsize=7, frameon=False)
add_panel_letter(ax, "D")
save_panel_figure(fig, "figure_4D")
plt.show()


## Panel E Processing

Fit global squared-decay estimates across the empirical population-size sweep. Bootstrap intervals are computed over starting genotypes to show sampling variability.


In [ ]:
def build_popsize_payload():
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    popsize_results = {}
    for name in EMPIRICAL_NAMES:
        curves_by_pop = empirical_popsize_curves[name]
        raw_params = empirical_popsize_payloads[name]["params"]
        pop_sizes = np.asarray(raw_params["pop_sizes"], dtype=int)
        rates = []
        for pop_index in tqdm(range(curves_by_pop.shape[0]), desc=f"{name} popsize"):
            curves = curves_by_pop[pop_index]
            start_gmu = curves.mean(axis=1) ** 2
            boot_vals = []
            sample_size = min(BOOTSTRAP_STARTS_FOR_GLOBAL_ESTIMATES, start_gmu.shape[0])
            for _ in range(POPSIZE_BOOTSTRAPS):
                idx = rng.choice(start_gmu.shape[0], size=sample_size, replace=False)
                boot_vals.append(fit_rho2_from_gmu(start_gmu[idx].mean(axis=0), steps=start_gmu.shape[-1]))
            rates.append(finite_array(boot_vals))
        popsize_results[name] = {"pop_sizes": pop_sizes, "rates": rates}
    return {
        "data": popsize_results,
        "metadata": {"paper_reference": "Figure 4E", "description": "Global fitted rho2 over empirical population-size sweeps."},
    }

popsize_payload = load_or_build_processed("popsize", build_popsize_payload)
popsize_data = popsize_payload["data"]
{name: len(payload["rates"]) for name, payload in popsize_data.items()}


### Panel E Individual Export
This cell saves the annotated Figure 4E subfigure after the population-size sensitivity analysis has been processed.


In [ ]:
fig, ax = plt.subplots(figsize=(3.3, 2.8), dpi=PANEL_DPI)
for n, name in enumerate(EMPIRICAL_NAMES):
    pop_sizes = np.asarray(popsize_data[name]["pop_sizes"])
    vals = popsize_data[name]["rates"]
    means = np.asarray([np.nanmean(v) for v in vals])
    stds = np.asarray([np.nanstd(v) for v in vals])
    ax.plot(pop_sizes, means, color=colours[n], marker=markers[n], markersize=4, linewidth=1, label=labels[n])
    ax.fill_between(pop_sizes, means - stds, means + stds, color=colours[n], alpha=0.18)
    ax.hlines(fourier_vals[n], xmin=pop_sizes.min(), xmax=pop_sizes.max(), color=colours[n], linestyle="dotted", linewidth=1)
ax.set_xscale("log")
ax.set_title("Population-size sensitivity")
ax.set_xlabel("Population size")
ax.set_ylabel(r"Global fitted $\rho_2$")
ax.legend(fontsize=7, frameon=False)
add_panel_letter(ax, "E")
save_panel_figure(fig, "figure_4E")
plt.show()


## Panel F Processing

Fit global squared-decay estimates while varying the number of sampled generations retained from the all-start decay curves.


In [ ]:
def build_generation_count_payload():
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    generation_results = {}
    for name in EMPIRICAL_NAMES:
        curves = empirical_all_curves[name]
        start_gmu = curves.mean(axis=1) ** 2
        n_steps = start_gmu.shape[-1]
        generation_counts = np.unique(np.clip(np.array([5, 10, 15, 20, n_steps]), 3, n_steps))
        landscape_rates = []
        sample_size = min(BOOTSTRAP_STARTS_FOR_GLOBAL_ESTIMATES, start_gmu.shape[0])
        for generation_count in tqdm(generation_counts, desc=f"{name} generation count"):
            boot_vals = []
            for _ in range(GENERATION_COUNT_BOOTSTRAPS):
                idx = rng.choice(start_gmu.shape[0], size=sample_size, replace=False)
                g_mu = start_gmu[idx, : int(generation_count)].mean(axis=0)
                boot_vals.append(fit_rho2_from_gmu(g_mu, steps=int(generation_count)))
            landscape_rates.append(finite_array(boot_vals))
        generation_results[name] = {"generation_counts": generation_counts, "rates": landscape_rates}
    return {
        "data": generation_results,
        "metadata": {"paper_reference": "Figure 4F", "description": "Global fitted rho2 over sampled generation counts."},
    }

generation_count_payload = load_or_build_processed("generation_count", build_generation_count_payload)
generation_count_data = generation_count_payload["data"]
{name: payload["generation_counts"] for name, payload in generation_count_data.items()}


### Panel F Individual Export
This cell saves the annotated Figure 4F subfigure after the generation-count sensitivity analysis has been processed.


In [ ]:
fig, ax = plt.subplots(figsize=(3.3, 2.8), dpi=PANEL_DPI)
for n, name in enumerate(EMPIRICAL_NAMES):
    counts = np.asarray(generation_count_data[name]["generation_counts"])
    vals = generation_count_data[name]["rates"]
    means = np.asarray([np.nanmean(v) for v in vals])
    stds = np.asarray([np.nanstd(v) for v in vals])
    ax.plot(counts, means, color=colours[n], marker=markers[n], markersize=4, linewidth=1, label=labels[n])
    ax.fill_between(counts, means - stds, means + stds, color=colours[n], alpha=0.18)
    ax.hlines(fourier_vals[n], xmin=counts.min(), xmax=counts.max(), color=colours[n], linestyle="dotted", linewidth=1)
ax.set_title("Generation-count sensitivity")
ax.set_xlabel("Generations used")
ax.set_ylabel(r"Global fitted $\rho_2$")
ax.legend(fontsize=7, frameon=False)
add_panel_letter(ax, "F")
save_panel_figure(fig, "figure_4F")
plt.show()


## Panel G Processing

Compute alternative empirical ruggedness metrics on the complete landscapes for the cross-metric comparison.


In [ ]:
def build_metric_comparison_payload():
    empirical_landscapes = [landscapes[name] for name in EMPIRICAL_NAMES]
    decay_rate_measurements = []
    for name in EMPIRICAL_NAMES:
        curves = empirical_all_curves[name]
        g_mu = np.mean(curves**2, axis=(0, 1))
        decay_rate_measurements.append(fit_rho2_from_gmu(g_mu, steps=g_mu.shape[-1]))

    roughness_values = [roughness_to_slope(landscape) for landscape in empirical_landscapes]
    landscape_r2_values = [1 - landscape_r2(landscape) for landscape in empirical_landscapes]
    dirichlet_values = [get_dirichlet_metric(landscape, on_gpu=False) for landscape in empirical_landscapes]
    entropy_values = [get_spectral_entropy(landscape, on_gpu=False) for landscape in empirical_landscapes]
    starting_points = [np.array([3, 17, 0, 3]), np.array([3, 8, 3, 18]), np.array([19, 17, 11, 18]), np.array([17, 12, 16])]
    epistasis_raw = [local_epistasis(landscape, start) for landscape, start in zip(empirical_landscapes, starting_points)]
    epistasis_values = [
        value.get("simple_sign_episasis", 0) + value.get("reciprocal_sign_epistasis", 0)
        if isinstance(value, dict)
        else value
        for value in epistasis_raw
    ]
    paths_to_max_values = [get_mean_paths_to_max(landscape, norm=False) for landscape in empirical_landscapes]
    paths_to_max_values[3] = paths_to_max_values[3] * (max_possible_paths(landscapes["GB1"].shape) / max_possible_paths(landscapes["ParD3"].shape))
    local_max_values = [find_distance_to_closest_max(landscape) for landscape in empirical_landscapes]

    return {
        "data": {
            "Decay rate": np.asarray(decay_rate_measurements),
            "Dirichlet": np.asarray(dirichlet_values),
            "Landscape R2": np.asarray(landscape_r2_values),
            "Spectral entropy": np.asarray(entropy_values),
            "Local epistasis": np.asarray(epistasis_values),
            "Distance to local max": np.asarray(local_max_values),
            "Paths to max": np.asarray(paths_to_max_values),
            "Roughness to slope": np.asarray(roughness_values),
        },
        "metadata": {"paper_reference": "Figure 4G", "description": "Empirical ruggedness metric comparison."},
    }

metric_payload = load_or_build_processed("metric_comparison", build_metric_comparison_payload)
metric_data = metric_payload["data"]
metric_data.keys()


### Panel G Individual Export
This cell saves the annotated Figure 4G subfigure after the empirical metric comparison has been processed.


In [ ]:
metric_names = list(metric_data.keys())
metric_values = [np.asarray(metric_data[name], dtype=float) for name in metric_names]
x_offsets = np.linspace(-0.27, 0.27, len(labels))
fig, ax = plt.subplots(figsize=(7.2, 3.0), dpi=PANEL_DPI)
for metric_index, values in enumerate(metric_values):
    span = np.nanmax(values) - np.nanmin(values)
    normalized = (values - np.nanmin(values)) / (span + EPS)
    for landscape_index, value in enumerate(normalized):
        ax.scatter(
            metric_index + x_offsets[landscape_index],
            value,
            color=colours[landscape_index],
            marker=markers[landscape_index],
            s=34,
            edgecolors="black",
            linewidth=0.4,
            label=labels[landscape_index] if metric_index == 0 else None,
        )
ax.set_xticks(np.arange(len(metric_names)))
ax.set_xticklabels(metric_names, rotation=30, ha="right")
ax.set_ylabel("Normalized ruggedness rank")
ax.set_ylim(-0.08, 1.08)
ax.set_title("Empirical ruggedness metric comparison")
ax.legend(fontsize=8, frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.32))
add_panel_letter(ax, "G")
save_panel_figure(fig, "figure_4G")
plt.show()


## Visualisation

Compose Figure 4 panels A-G from the processed payloads. The output uses the paper's empirical landscape colour order: GB1, TrpB, TEV, ParD3.


In [ ]:
# Figure styling.
labelsize = 8
ticksize = 6
titlesize = 10
legendsize = 7
dpi = 350
plt.rcParams["font.family"] = "DejaVu Sans"

fig = plt.figure(figsize=(9.0, 8.5), dpi=dpi, constrained_layout=True)
gs = GridSpec(3, 3, figure=fig, height_ratios=[1.0, 1.0, 1.05])
axes = {
    "A": fig.add_subplot(gs[0, 0]),
    "B": fig.add_subplot(gs[0, 1]),
    "C": fig.add_subplot(gs[0, 2]),
    "D": fig.add_subplot(gs[1, 0]),
    "E": fig.add_subplot(gs[1, 1]),
    "F": fig.add_subplot(gs[1, 2]),
    "G": fig.add_subplot(gs[2, :]),
}


def add_panel_letter(ax, letter: str) -> None:
    ax.text(-0.16, 1.08, letter, transform=ax.transAxes, fontsize=12, fontweight="bold", va="top", ha="left")


# Panel A: Fourier spectra.
ax = axes["A"]
for n, spectrum in enumerate(spectra_data["spectra"]):
    nonconstant = np.asarray(spectrum, dtype=float)[1:]
    spectrum_norm = (nonconstant - nonconstant.min()) / (nonconstant.max() - nonconstant.min() + EPS)
    indexes = np.arange(1, len(nonconstant) + 1)
    ax.plot(indexes, spectrum_norm, marker=markers[n], markersize=3, linewidth=1, color=colours[n], label=labels[n])
    ax.axvline(spectra_data["frequency_centroids"][n], color=colours[n], linestyle="--", alpha=0.55, linewidth=1)
ax.set_title("Empirical Fourier spectra", fontsize=titlesize)
ax.set_xlabel(r"Frequency index $i$", fontsize=labelsize)
ax.set_ylabel(r"Norm. power $b_i$", fontsize=labelsize)
ax.set_xticks([1, 2, 3, 4])
ax.tick_params(axis="both", labelsize=ticksize)
ax.legend(fontsize=legendsize - 1, frameon=False)
add_panel_letter(ax, "A")

# Panel B: local fitted rho2 distributions.
ax = axes["B"]
local_data = [finite_array(local_rhos[name]) for name in EMPIRICAL_NAMES]
vp = ax.violinplot(local_data, showmeans=True, showextrema=False)
for body, color in zip(vp["bodies"], colours):
    body.set_facecolor(color)
    body.set_edgecolor("black")
    body.set_alpha(0.55)
vp["cmeans"].set_color("black")
ax.scatter(np.arange(1, 5), fourier_vals, color="black", s=18, zorder=3, label=r"Analytical $\rho_2$")
ax.set_xticks(np.arange(1, 5))
ax.set_xticklabels(labels, rotation=25, ha="right", fontsize=ticksize)
ax.set_ylabel(r"Local fitted $\rho_2$", fontsize=labelsize)
ax.set_title("Local ruggedness estimates", fontsize=titlesize)
ax.tick_params(axis="y", labelsize=ticksize)
ax.legend(fontsize=legendsize - 1, frameon=False)
add_panel_letter(ax, "B")

# Panel C: replicate variability.
ax = axes["C"]
variability_data = [finite_array(replicate_variability[name]) for name in EMPIRICAL_NAMES]
vp = ax.violinplot(variability_data, showmeans=True, showextrema=False)
for body, color in zip(vp["bodies"], colours):
    body.set_facecolor(color)
    body.set_edgecolor("black")
    body.set_alpha(0.55)
vp["cmeans"].set_color("black")
for i, values in enumerate(variability_data, start=1):
    ax.hlines(np.nanmean(values), i - 0.28, i + 0.28, colors="black", linewidth=1.2)
ax.set_xticks(np.arange(1, 5))
ax.set_xticklabels(labels, rotation=25, ha="right", fontsize=ticksize)
ax.set_ylabel(r"s.d. of fitted $\rho_2$", fontsize=labelsize)
ax.set_title("Replicate uncertainty", fontsize=titlesize)
ax.tick_params(axis="y", labelsize=ticksize)
add_panel_letter(ax, "C")

# Panel D: subsampling trajectory counts.
ax = axes["D"]
for n, name in enumerate(EMPIRICAL_NAMES):
    counts = np.asarray(trajectory_counts[name])
    vals = subsampling_data[name]
    means = np.asarray([np.nanmean(v) for v in vals])
    stds = np.asarray([np.nanstd(v) for v in vals])
    ax.plot(counts, means, color=colours[n], marker=markers[n], markersize=3, linewidth=1, label=labels[n])
    ax.fill_between(counts, means - stds, means + stds, color=colours[n], alpha=0.18)
    ax.hlines(fourier_vals[n], xmin=counts.min(), xmax=counts.max(), color=colours[n], linestyle="dotted", linewidth=1)
ax.set_xscale("log")
ax.set_title("Starting-point subsampling", fontsize=titlesize)
ax.set_xlabel("Number of starting points", fontsize=labelsize)
ax.set_ylabel(r"Global fitted $\rho_2$", fontsize=labelsize)
ax.tick_params(axis="both", labelsize=ticksize)
add_panel_letter(ax, "D")

# Panel E: population size.
ax = axes["E"]
for n, name in enumerate(EMPIRICAL_NAMES):
    pop_sizes = np.asarray(popsize_data[name]["pop_sizes"])
    vals = popsize_data[name]["rates"]
    means = np.asarray([np.nanmean(v) for v in vals])
    stds = np.asarray([np.nanstd(v) for v in vals])
    ax.plot(pop_sizes, means, color=colours[n], marker=markers[n], markersize=3, linewidth=1, label=labels[n])
    ax.fill_between(pop_sizes, means - stds, means + stds, color=colours[n], alpha=0.18)
    ax.hlines(fourier_vals[n], xmin=pop_sizes.min(), xmax=pop_sizes.max(), color=colours[n], linestyle="dotted", linewidth=1)
ax.set_xscale("log")
ax.set_title("Population-size sensitivity", fontsize=titlesize)
ax.set_xlabel("Population size", fontsize=labelsize)
ax.set_ylabel(r"Global fitted $\rho_2$", fontsize=labelsize)
ax.tick_params(axis="both", labelsize=ticksize)
add_panel_letter(ax, "E")

# Panel F: sampled generation count.
ax = axes["F"]
for n, name in enumerate(EMPIRICAL_NAMES):
    counts = np.asarray(generation_count_data[name]["generation_counts"])
    vals = generation_count_data[name]["rates"]
    means = np.asarray([np.nanmean(v) for v in vals])
    stds = np.asarray([np.nanstd(v) for v in vals])
    ax.plot(counts, means, color=colours[n], marker=markers[n], markersize=3, linewidth=1, label=labels[n])
    ax.fill_between(counts, means - stds, means + stds, color=colours[n], alpha=0.18)
    ax.hlines(fourier_vals[n], xmin=counts.min(), xmax=counts.max(), color=colours[n], linestyle="dotted", linewidth=1)
ax.set_title("Generation-count sensitivity", fontsize=titlesize)
ax.set_xlabel("Generations used", fontsize=labelsize)
ax.set_ylabel(r"Global fitted $\rho_2$", fontsize=labelsize)
ax.tick_params(axis="both", labelsize=ticksize)
add_panel_letter(ax, "F")

# Panel G: metric comparison.
ax = axes["G"]
metric_names = list(metric_data.keys())
metric_values = [np.asarray(metric_data[name], dtype=float) for name in metric_names]
x_offsets = np.linspace(-0.27, 0.27, len(labels))
for metric_index, values in enumerate(metric_values):
    span = np.nanmax(values) - np.nanmin(values)
    normalized = (values - np.nanmin(values)) / (span + EPS)
    for landscape_index, value in enumerate(normalized):
        ax.scatter(metric_index + x_offsets[landscape_index], value, color=colours[landscape_index], marker=markers[landscape_index], s=30, edgecolors="black", linewidth=0.4, label=labels[landscape_index] if metric_index == 0 else None)
ax.set_xticks(np.arange(len(metric_names)))
ax.set_xticklabels(metric_names, rotation=30, ha="right", fontsize=ticksize)
ax.set_ylabel("Normalized ruggedness rank", fontsize=labelsize)
ax.set_ylim(-0.08, 1.08)
ax.set_title("Empirical ruggedness metric comparison", fontsize=titlesize)
ax.tick_params(axis="y", labelsize=ticksize)
ax.legend(fontsize=legendsize, frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.32))
add_panel_letter(ax, "G")

if SAVE_FIGURES:
    for figure_type in ("pdf", "png", "eps"):
        fig.savefig(FIGURES_DIR / figure_type / f"figure_4.{figure_type}", dpi=dpi, bbox_inches="tight")
plt.show()
